# LlamaIndex 進階索引技術

## 📚 學習目標

通過本教程，你將學習：
1. 多種高級索引結構的原理和使用
2. 索引優化技巧
3. 分層索引策略
4. 向量存儲優化
5. 實際生產環境的最佳實踐

**難度**: ⭐⭐ 進階  
**預計時間**: 1-1.5 小時

## 環境設置

In [ ]:
# 安裝必要的套件
!pip install llama-index llama-index-llms-openai llama-index-embeddings-openai -q
!pip install llama-index-vector-stores-chroma chromadb -q

In [ ]:
import os
from dotenv import load_dotenv

# 加載環境變數
load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

print("✅ 環境設置完成")

## 準備示例數據

In [ ]:
# 創建測試數據
import os

os.makedirs("data", exist_ok=True)

sample_data = {
    "python_basics.txt": """Python 是一種高級程式語言，以其簡潔和可讀性著稱。
    Python 支援多種程式設計範式，包括物件導向、函數式和程序式程式設計。
    Python 擁有豐富的標準庫和第三方套件生態系統。""",
    
    "ml_frameworks.txt": """機器學習領域有許多流行的框架：
    - TensorFlow: Google 開發的深度學習框架
    - PyTorch: Facebook 開發的深度學習框架
    - scikit-learn: 傳統機器學習算法庫
    - Keras: 高級神經網絡 API""",
    
    "data_science.txt": """數據科學結合了統計學、機器學習和領域知識。
    常用工具包括 Pandas（數據處理）、NumPy（數值計算）、Matplotlib（可視化）。
    數據科學流程包括：數據收集、清洗、探索、建模、部署。"""
}

for filename, content in sample_data.items():
    with open(f"data/{filename}", "w", encoding="utf-8") as f:
        f.write(content)

print(f"✅ 已創建 {len(sample_data)} 個測試文件")

## 1. 樹形索引 (Tree Index)

樹形索引通過自下而上的總結構建索引樹，適合需要層次化信息組織的場景。

### 使用場景
- 階層性文檔（如組織架構、產品目錄）
- 長文檔（需要先總覽再深入細節）
- 多層次查詢（需要在不同抽象層次回答問題）

In [ ]:
from llama_index.core import TreeIndex, SimpleDirectoryReader

# 加載文檔
documents = SimpleDirectoryReader("data").load_data()
print(f"📄 已加載 {len(documents)} 個文檔")

# 創建樹形索引
print("\n🌳 創建樹形索引...")
tree_index = TreeIndex.from_documents(
    documents,
    num_children=10,  # 每個父節點的子節點數
    build_tree=True
)

# 查詢
query_engine = tree_index.as_query_engine(
    child_branch_factor=2  # 查詢時探索的分支數
)

response = query_engine.query("Python 的主要特點是什麼？")
print(f"\n回答: {response}")

### 樹形索引的優缺點

**優點**:
- ✅ 支持多層次查詢
- ✅ 可以從高層次總覽到細節
- ✅ 適合結構化文檔

**缺點**:
- ❌ 構建時間較長
- ❌ 對扁平文檔效果一般
- ❌ 查詢速度可能較慢

## 2. 關鍵詞索引 (Keyword Table Index)

基於關鍵詞的索引，通過提取關鍵詞進行快速匹配。

### 使用場景
- 精確匹配查詢（已知關鍵詞）
- 多關鍵詞搜索
- 快速過濾大量文檔

In [ ]:
from llama_index.core import KeywordTableIndex

# 創建關鍵詞索引
print("🔑 創建關鍵詞索引...")
keyword_index = KeywordTableIndex.from_documents(
    documents,
    max_keywords_per_chunk=10  # 每個文檔塊提取的最大關鍵詞數
)

# 查詢
query_engine = keyword_index.as_query_engine()
response = query_engine.query("TensorFlow PyTorch")
print(f"\n回答: {response}")

## 3. 混合索引策略

組合多個索引，發揮各自優勢。先用簡單索引過濾，再用複雜索引精確查詢。

In [ ]:
from llama_index.core import VectorStoreIndex, SummaryIndex
from llama_index.core.tools import QueryEngineTool
from llama_index.core.query_engine import RouterQueryEngine
from llama_index.core.selectors import LLMSingleSelector

# 創建多個索引
print("📊 創建多個索引...")
vector_index = VectorStoreIndex.from_documents(documents)
summary_index = SummaryIndex.from_documents(documents)

# 創建查詢引擎工具
vector_tool = QueryEngineTool.from_defaults(
    query_engine=vector_index.as_query_engine(),
    description="用於語義搜索和相似度查詢，適合回答具體問題"
)

summary_tool = QueryEngineTool.from_defaults(
    query_engine=summary_index.as_query_engine(),
    description="用於總結性查詢，適合回答需要綜合所有文檔的問題"
)

# 創建路由查詢引擎
router_query_engine = RouterQueryEngine(
    selector=LLMSingleSelector.from_defaults(),
    query_engine_tools=[vector_tool, summary_tool]
)

# 測試 - 具體問題（會選擇向量索引）
print("\n問題 1: Python 的主要特點是什麼？")
response1 = router_query_engine.query("Python 的主要特點是什麼？")
print(f"回答: {response1}")

# 測試 - 總結性問題（會選擇摘要索引）
print("\n" + "="*80)
print("問題 2: 總結所有文檔的主要內容")
response2 = router_query_engine.query("總結所有文檔的主要內容")
print(f"回答: {response2}")

## 4. 向量存儲優化 - 使用 Chroma

使用專業向量資料庫可以提升性能和可擴展性。

In [ ]:
import chromadb
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core import StorageContext

# 創建 Chroma 客戶端
print("🗄️ 初始化 Chroma 向量資料庫...")
chroma_client = chromadb.PersistentClient(path="./chroma_db")

# 創建或獲取集合
collection_name = "llamaindex_demo"
try:
    chroma_client.delete_collection(collection_name)
except:
    pass

chroma_collection = chroma_client.create_collection(collection_name)

# 創建向量存儲
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

# 創建索引
index = VectorStoreIndex.from_documents(
    documents,
    storage_context=storage_context
)

print("✅ 索引已創建並存儲在 Chroma 中")

# 查詢
query_engine = index.as_query_engine()
response = query_engine.query("機器學習框架有哪些？")
print(f"\n回答: {response}")

### 載入已存在的 Chroma 索引

In [ ]:
# 下次可以直接加載
print("📂 從 Chroma 加載索引...")

chroma_client = chromadb.PersistentClient(path="./chroma_db")
chroma_collection = chroma_client.get_collection(collection_name)

vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
loaded_index = VectorStoreIndex.from_vector_store(vector_store=vector_store)

print("✅ 索引加載完成")

# 使用加載的索引
query_engine = loaded_index.as_query_engine()
response = query_engine.query("數據科學的常用工具有哪些？")
print(f"\n回答: {response}")

## 5. 索引性能優化

### 增量更新索引

In [ ]:
from llama_index.core import Document

# 插入新文檔
new_doc = Document(text="LlamaIndex 是一個專注於數據索引的 LLM 框架，特別適合構建 RAG 應用。")

print("➕ 插入新文檔...")
index.insert(new_doc)

# 查詢新加入的內容
query_engine = index.as_query_engine()
response = query_engine.query("LlamaIndex 是什麼？")
print(f"\n回答: {response}")

# 刪除文檔
# index.delete_ref_doc(ref_doc_id="doc_id", delete_from_docstore=True)

## 6. 自定義分塊策略

In [ ]:
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core import Settings

# 配置自定義分塊器
Settings.node_parser = SentenceSplitter(
    chunk_size=512,      # 每塊的大小
    chunk_overlap=50,    # 重疊大小
    separator=" "        # 分隔符
)

# 使用自定義分塊創建索引
custom_index = VectorStoreIndex.from_documents(documents)

print("✅ 使用自定義分塊策略創建索引完成")

# 查看節點信息
nodes = custom_index.docstore.docs
print(f"\n總節點數: {len(nodes)}")
print(f"第一個節點的文本長度: {len(list(nodes.values())[0].text)} 字符")

## 7. 實際應用案例：混合檢索策略

結合向量檢索和關鍵詞檢索，提高檢索準確性。

In [ ]:
from llama_index.core.retrievers import VectorIndexRetriever, KeywordTableSimpleRetriever
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core import QueryBundle

# 創建兩種檢索器
vector_retriever = VectorIndexRetriever(
    index=vector_index,
    similarity_top_k=5
)

keyword_retriever = KeywordTableSimpleRetriever(
    index=keyword_index
)

# 自定義混合檢索器
class HybridRetriever:
    """混合檢索器：結合向量檢索和關鍵詞檢索"""
    
    def __init__(self, vector_retriever, keyword_retriever):
        self.vector_retriever = vector_retriever
        self.keyword_retriever = keyword_retriever
    
    def retrieve(self, query_bundle: QueryBundle):
        # 獲取兩種檢索結果
        vector_nodes = self.vector_retriever.retrieve(query_bundle)
        keyword_nodes = self.keyword_retriever.retrieve(query_bundle)
        
        # 合併並去重
        all_nodes = vector_nodes + keyword_nodes
        unique_nodes = {node.node_id: node for node in all_nodes}.values()
        
        # 重新排序
        sorted_nodes = sorted(
            unique_nodes,
            key=lambda x: x.score if hasattr(x, 'score') else 0,
            reverse=True
        )
        
        return sorted_nodes[:5]  # 返回前5個

# 使用混合檢索器
hybrid_retriever = HybridRetriever(vector_retriever, keyword_retriever)
query_engine = RetrieverQueryEngine(retriever=hybrid_retriever)

# 測試
response = query_engine.query("Python 機器學習框架")
print(f"混合檢索回答: {response}")

## 📝 本教程總結

### ✅ 已掌握的技能

1. **多種索引類型**:
   - ✅ TreeIndex（樹形索引）- 適合階層性文檔
   - ✅ KeywordTableIndex（關鍵詞索引）- 適合精確匹配
   - ✅ VectorStoreIndex（向量索引）- 通用語義搜索

2. **高級技術**:
   - ✅ 路由查詢引擎（自動選擇最佳索引）
   - ✅ 混合檢索策略（結合多種檢索方法）
   - ✅ 向量資料庫整合（Chroma）

3. **優化技巧**:
   - ✅ 增量更新索引
   - ✅ 自定義分塊策略
   - ✅ 性能調優

### 🎯 索引選擇指南

| 場景 | 推薦索引 | 原因 |
|------|---------|------|
| 一般問答 | VectorStoreIndex | 平衡性能和準確性 |
| 長文檔摘要 | TreeIndex | 層次化組織 |
| 精確匹配 | KeywordTableIndex | 快速關鍵詞查找 |
| 總結性問題 | SummaryIndex | 全文檔掃描 |
| 混合查詢 | Router + 多索引 | 發揮各自優勢 |

### 💡 生產環境建議

1. **使用專業向量資料庫**：Chroma、Pinecone、Weaviate
2. **實現增量更新**：避免全量重建索引
3. **合理設置分塊**：chunk_size=512-1024, overlap=10-20%
4. **混合檢索策略**：結合向量和關鍵詞檢索
5. **監控和評估**：記錄查詢性能和準確率

### 🚀 下一步

- **2.數據加載與處理.ipynb**: 學習 100+ 種數據加載器
- **3.查詢引擎深入.ipynb**: 深入理解查詢優化
- **4.Chat_Engine聊天引擎.ipynb**: 構建對話系統